# Stock snapshot

Writes the end-of-day stock per store and product for the ordering team.

In [ ]:
snapshot_date = "2026-09-30"
month_suffix = snapshot_date[:7].replace("-", "_")

In [ ]:
levels = spark.read.table("erp.stock_levels")
stores = spark.table("dim_store")
products = spark.sql("SELECT product_id, category FROM marts.dim_product")
adjustments = spark.sql(f"SELECT store_id, product_id, qty_delta FROM erp.stock_adjustments_{month_suffix}")

In [ ]:
from pyspark.sql import functions as F

snapshot = (
    levels.join(stores, "store_id")
    .join(products, "product_id")
    .join(adjustments, ["store_id", "product_id"], "left")
    .withColumn("on_hand", F.col("quantity") + F.coalesce(F.col("qty_delta"), F.lit(0)))
    .withColumn("snapshot_date", F.lit(snapshot_date))
    .select("snapshot_date", "store_id", "region", "product_id", "category", "on_hand")
)

snapshot.write.mode("overwrite").saveAsTable("marts.stock_snapshot")